<a href="https://colab.research.google.com/github/arihant-jaggi/stock-forecast/blob/main/StockForecastingWalkForwardWilcoxonEtc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance tensorflow scikit-learn scipy pandas numpy matplotlib -q

import numpy as np
import pandas as pd
import yfinance as yf
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.tree import DecisionTreeRegressor
from scipy.stats import wilcoxon
import random


np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

TICKERS = ["NKE", "AAPL", "NVDA", "JNJ", "XOM", "JPM", "MSFT", "AMZN",
           "GOOGL", "META", "TSLA", "V", "MA", "PG", "HD", "MRVL",
           "AVGO", "F", "NFLX", "WMT", "AMD"]

START_DATE = "2015-01-01"
END_DATE   = "2026-02-17"
LOOKBACK   = 60
HORIZON    = 30
TRAIN_FRAC = 0.80
VAL_FRAC   = 0.10

N_WINDOWS  = 3

In [ ]:
def fetch_stock_data(ticker, start=START_DATE, end=END_DATE):
    df = yf.download(ticker, start=start, end=end, auto_adjust=True, progress=False)
    if df.empty:
        raise ValueError(f"No data returned for {ticker}")
    df = df.dropna()
    prices  = df["Close"].values.astype(float).flatten()
    volumes = df["Volume"].values.astype(float).flatten()
    return prices, volumes


FEATURE_KEYS = ["logReturn", "return5d", "return10d", "rollingMean20",
                "rollingStd20", "rsi14", "macd", "macdSignal", "volumeRatio"]

def _ema(arr, span, idx):
    k = 2.0 / (span + 1.0)
    val = arr[0]
    for i in range(1, min(idx, len(arr) - 1) + 1):
        val = arr[i] * k + val * (1 - k)
    return val

def compute_features(prices, volumes):
    n = len(prices)
    rows = []

    log_returns = [0.0]
    for i in range(1, n):
        log_returns.append(np.log(prices[i] / prices[i - 1]))

    for i in range(n):
        row = {}
        row["price"]     = prices[i]
        row["logReturn"] = log_returns[i]
        row["return5d"]  = (prices[i] - prices[i - 5]) / prices[i - 5] if i >= 5 else 0.0
        row["return10d"] = (prices[i] - prices[i - 10]) / prices[i - 10] if i >= 10 else 0.0
        if i >= 19:
            window = prices[i - 19:i + 1]
            mean = window.mean()
            std  = np.sqrt(((window - mean) ** 2).mean())
            row["rollingMean20"] = mean
            row["rollingStd20"]  = std
        else:
            row["rollingMean20"] = prices[i]
            row["rollingStd20"]  = 0.0
        if i >= 14:
            gain_sum = 0.0
            loss_sum = 0.0
            for j in range(i - 13, i + 1):
                diff = prices[j] - prices[j - 1]
                if diff > 0:
                    gain_sum += diff
                else:
                    loss_sum -= diff
            avg_gain = gain_sum / 14
            avg_loss = loss_sum / 14 + 1e-10
            row["rsi14"] = 100 - 100 / (1 + avg_gain / avg_loss)
        else:
            row["rsi14"] = 50.0
        row["ema12"] = _ema(prices, 12, i)
        row["ema26"] = _ema(prices, 26, i)
        row["macd"]  = row["ema12"] - row["ema26"]
        if volumes is not None and len(volumes) == n and i >= 19:
            v_window = volumes[i - 19:i + 1]
            v_mean = v_window.mean()
            row["volumeRatio"] = volumes[i] / v_mean if v_mean > 0 else 1.0
        else:
            row["volumeRatio"] = 1.0
        rows.append(row)

    macds = np.array([r["macd"] for r in rows])
    for i in range(len(rows)):
        rows[i]["macdSignal"] = _ema(macds, 9, i)

    return rows

def feature_vector(row):
    return [row.get(k, 0.0) for k in FEATURE_KEYS]


def time_split(arr, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC):
    n = len(arr)
    train_end = int(n * train_frac)
    val_end   = int(n * (train_frac + val_frac))
    return arr[:train_end], arr[train_end:val_end], arr[val_end:]

class MinMaxScaler:
    def __init__(self, data):
        self.min = float(np.min(data))
        self.max = float(np.max(data))
        self.range = (self.max - self.min) or 1.0
    def transform(self, arr):
        return [(v - self.min) / self.range for v in arr]
    def inverse(self, arr):
        return [v * self.range + self.min for v in arr]

def build_sequences(scaled_arr, lookback=LOOKBACK):
    X, y = [], []
    for i in range(lookback, len(scaled_arr)):
        X.append([[v] for v in scaled_arr[i - lookback:i]])
        y.append(scaled_arr[i])
    return np.array(X), np.array(y)


def _metrics(preds, actuals):
    err = np.abs(preds - actuals)
    mae  = err.mean()
    rmse = np.sqrt((err ** 2).mean())
    mape = (err / (np.abs(actuals) + 1e-10)).mean() * 100
    return {"MAE": round(float(mae), 4),
            "RMSE": round(float(rmse), 4),
            "MAPE": round(float(mape), 4)}

In [ ]:
def fit_gbm(prices):
    returns = np.diff(np.log(prices))
    mean = returns.mean()
    std  = returns.std()
    mu    = mean * 252
    sigma = std * np.sqrt(252)
    return {"mu": mu, "sigma": sigma, "lastPrice": prices[-1]}

def forecast_gbm(params, horizon=HORIZON, n_paths=5000, seed=42):
    rng = np.random.RandomState(seed)
    mu, sigma, last_price = params["mu"], params["sigma"], params["lastPrice"]
    fp = horizon
    dt = 1.0 / fp
    time_axis = np.array([i / fp for i in range(fp + 1)])

    paths = np.zeros((n_paths, fp + 1))
    for p in range(n_paths):
        b = rng.standard_normal(fp) * np.sqrt(dt)
        W = np.cumsum(b)
        paths[p, 0] = last_price
        for t in range(1, fp + 1):
            drift     = (mu - 0.5 * sigma * sigma) * time_axis[t]
            diffusion = sigma * W[t - 1]
            paths[p, t] = last_price * np.exp(drift + diffusion)

    median  = np.median(paths[:, 1:], axis=0)
    lower5  = np.percentile(paths[:, 1:], 5,  axis=0)
    upper95 = np.percentile(paths[:, 1:], 95, axis=0)
    return {"median": median, "lower5": lower5, "upper95": upper95}

def evaluate_gbm(params, test_prices, context_last_price):
    preds = []
    for i in range(len(test_prices)):
        price = context_last_price if i == 0 else test_prices[i - 1]
        p = {**params, "lastPrice": price}
        median = forecast_gbm(p, horizon=1, n_paths=500, seed=42 + i)["median"]
        preds.append(median[0])
    return _metrics(np.array(preds), np.array(test_prices))


def build_lstm_model(lookback=LOOKBACK):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(lookback, 1)),
        Dropout(0.2),
        LSTM(64, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1),
    ])
    model.compile(optimizer="adam", loss="mean_squared_error")
    return model

def train_lstm(train_prices, val_prices, lookback=LOOKBACK):
    scaler = MinMaxScaler(train_prices)
    train_scaled = scaler.transform(train_prices)

    combined = list(train_prices[-lookback:]) + list(val_prices)
    combined_scaled = scaler.transform(combined)

    x_train, y_train = build_sequences(train_scaled, lookback)
    x_val,   y_val   = build_sequences(combined_scaled, lookback)

    model = build_lstm_model(lookback)
    es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    model.fit(x_train, y_train,
              validation_data=(x_val, y_val),
              epochs=50, batch_size=32, verbose=0, callbacks=[es])
    return model, scaler

def forecast_lstm(model, scaler, recent_prices, lookback=LOOKBACK, horizon=HORIZON):
    scaled = scaler.transform(recent_prices[-lookback:])
    window = list(scaled)
    preds_scaled = []
    for _ in range(horizon):
        inp = np.array(window[-lookback:]).reshape(1, lookback, 1)
        val = float(model.predict(inp, verbose=0)[0][0])
        preds_scaled.append(val)
        window.append(val)
    return scaler.inverse(preds_scaled)

def evaluate_lstm(model, scaler, test_prices, context_prices, lookback=LOOKBACK):
    full = list(context_prices[-lookback:]) + list(test_prices)
    full_scaled = scaler.transform(full)
    x_test, y_test = build_sequences(full_scaled, lookback)
    preds_scaled = model.predict(x_test, verbose=0).flatten()
    preds   = np.array(scaler.inverse(preds_scaled.tolist()))
    actuals = np.array(scaler.inverse(y_test.tolist()))
    return _metrics(preds, actuals)


DT_GRID = {
    "max_depth":         [3, 5, 8, 12, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf":  [1, 2, 5, 10],
    "max_features":      [None, "sqrt", "log2"],
}

def _all_dt_configs():
    configs = []
    for md in DT_GRID["max_depth"]:
        for mss in DT_GRID["min_samples_split"]:
            for msl in DT_GRID["min_samples_leaf"]:
                for mf in DT_GRID["max_features"]:
                    configs.append(dict(max_depth=md, min_samples_split=mss,
                                        min_samples_leaf=msl, max_features=mf))
    return configs

def _build_supervised(prices, volumes):
    feats = compute_features(prices, volumes)
    X, y = [], []
    for i in range(len(feats) - 1):
        X.append(feature_vector(feats[i]))
        y.append(prices[i + 1])
    return np.array(X), np.array(y)

def train_decision_tree(train_prices, train_volumes, val_prices, val_volumes):
    X_train, y_train = _build_supervised(train_prices, train_volumes)
    X_val,   y_val   = _build_supervised(val_prices, val_volumes)

    configs = _all_dt_configs()
    max_configs = min(len(configs), 50)
    sampled = random.sample(configs, max_configs)

    best_rmse, best_tree = np.inf, None
    for c in sampled:
        tree = DecisionTreeRegressor(random_state=42, **c)
        tree.fit(X_train, y_train)
        preds = tree.predict(X_val)
        rmse = np.sqrt(np.mean((preds - y_val) ** 2))
        if rmse < best_rmse:
            best_rmse, best_tree = rmse, tree
    return best_tree

def evaluate_decision_tree(tree, test_prices, test_volumes):
    feats = compute_features(test_prices, test_volumes)
    preds, actuals = [], []
    for i in range(len(feats) - 1):
        preds.append(tree.predict([feature_vector(feats[i])])[0])
        actuals.append(test_prices[i + 1])
    return _metrics(np.array(preds), np.array(actuals))


def evaluate_naive(test_prices, context_last_price):
    preds, actuals = [], []
    for i in range(len(test_prices)):
        prev = context_last_price if i == 0 else test_prices[i - 1]
        preds.append(prev)
        actuals.append(test_prices[i])
    return _metrics(np.array(preds), np.array(actuals))


def walk_forward_evaluate(ticker):
    prices, volumes = fetch_stock_data(ticker)
    n = len(prices)

    last_region_start = int(n * 0.80)
    latest_start = n - HORIZON
    if N_WINDOWS == 1:
        window_starts = [latest_start]
    else:
        window_starts = np.linspace(last_region_start, latest_start,
                                    N_WINDOWS, dtype=int).tolist()

    per_window = {"GBM": [], "LSTM": [], "DT": [], "Naive": []}

    for w_idx, test_start in enumerate(window_starts):
        test_end = test_start + HORIZON
        hist_prices  = prices[:test_start]
        hist_volumes = volumes[:test_start]
        test_prices  = prices[test_start:test_end]

        tr_p, val_p, _ = time_split(hist_prices)
        tr_v, val_v, _ = time_split(hist_volumes)
        context_last_price = hist_prices[-1]

        gbm_params = fit_gbm(tr_p)
        per_window["GBM"].append(evaluate_gbm(gbm_params, test_prices, context_last_price))

        model, scaler = train_lstm(tr_p, val_p)
        per_window["LSTM"].append(evaluate_lstm(model, scaler, test_prices, hist_prices))

        tree = train_decision_tree(tr_p, tr_v, val_p, val_v)
        dt_test_prices  = np.concatenate([hist_prices[-40:], test_prices])
        dt_test_volumes = np.concatenate([hist_volumes[-40:], volumes[test_start:test_end]])
        per_window["DT"].append(evaluate_decision_tree(tree, dt_test_prices, dt_test_volumes))

        per_window["Naive"].append(evaluate_naive(test_prices, context_last_price))

        print(f"  {ticker}  window {w_idx+1}/{len(window_starts)} done")

    def _avg(list_of_dicts):
        return {m: round(float(np.mean([d[m] for d in list_of_dicts])), 4)
                for m in ["MAE", "RMSE", "MAPE"]}

    return {model: _avg(per_window[model]) for model in per_window}

In [ ]:

def fixed_window_naive(ticker):
    prices, volumes = fetch_stock_data(ticker)
    _, _, test_prices = time_split(prices)
    n = len(prices)
    val_end = int(n * (TRAIN_FRAC + VAL_FRAC))
    context_last_price = prices[val_end - 1]
    return evaluate_naive(test_prices, context_last_price)

print("Fixed-window Naive baseline (matches original Table 1 setup):\n")
print(f"{'Ticker':8s}{'Naive_MAE':>12s}{'Naive_RMSE':>12s}{'Naive_MAPE':>12s}")
rows = []
for ticker in TICKERS:
    try:
        m = fixed_window_naive(ticker)
        rows.append({"Ticker": ticker, **{f"Naive_{k}": v for k, v in m.items()}})
        print(f"{ticker:8s}{m['MAE']:>12.4f}{m['RMSE']:>12.4f}{m['MAPE']:>12.4f}")
    except Exception as e:
        print(f"{ticker:8s}  failed: {e}")

import pandas as pd
pd.DataFrame(rows).to_csv("naive_fixed_window.csv", index=False)
print("\nSaved → naive_fixed_window.csv")

Fixed-window Naive baseline (matches original Table 1 setup):

Ticker     Naive_MAE  Naive_RMSE  Naive_MAPE
NKE           1.1162      1.6338      1.7255
AAPL          2.8680      4.3248      1.2668
NVDA          3.1427      4.2671      2.1276
JNJ           1.4531      2.0443      0.8579
XOM           1.2897      1.7197      1.1610
JPM           3.0707      4.2395      1.1274
MSFT          4.8367      7.2435      1.0857
AMZN          3.2654      4.5564      1.5164
GOOGL         3.1187      4.2429      1.4455
META         10.5891     15.4245      1.6273
TSLA          9.7679     12.6752      2.8124
V             3.3385      4.8379      0.9889
MA            5.4902      7.8352      1.0045
PG            1.3953      1.8480      0.9074
HD            3.9475      5.2399      1.0818
MRVL          2.2877      3.4052      2.9066
AVGO          5.9165      8.7631      2.1968
F             0.1570      0.2193      1.4637
NFLX          1.5612      2.1897      1.4871
WMT           1.1058      1.5748     

In [ ]:
test_result = walk_forward_evaluate("AAPL")
print("\nAAPL result:")
for model, m in test_result.items():
    print(f"  {model:6s}  MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  MAPE={m['MAPE']:.4f}%")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  AAPL  window 1/3 done


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  AAPL  window 2/3 done


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  AAPL  window 3/3 done

AAPL result:
  GBM     MAE=48.5816  RMSE=49.0376  MAPE=21.3745%
  LSTM    MAE=16.6058  RMSE=18.1751  MAPE=6.6715%
  DT      MAE=54.1531  RMSE=54.9933  MAPE=22.1508%
  Naive   MAE=2.3989  RMSE=3.3284  MAPE=1.0175%


In [ ]:
def run_all():
    results = {}
    for ticker in TICKERS:
        print(f"Evaluating {ticker} ...")
        try:
            results[ticker] = walk_forward_evaluate(ticker)
        except Exception as e:
            print(f"  !! {ticker} failed: {e}")
    return results

def results_to_dataframe(results):
    rows = []
    for ticker, r in results.items():
        rows.append({
            "Ticker": ticker,
            "LSTM_MAE": r["LSTM"]["MAE"], "LSTM_RMSE": r["LSTM"]["RMSE"], "LSTM_MAPE": r["LSTM"]["MAPE"],
            "GBM_MAE":  r["GBM"]["MAE"],  "GBM_RMSE":  r["GBM"]["RMSE"],  "GBM_MAPE":  r["GBM"]["MAPE"],
            "DT_MAE":   r["DT"]["MAE"],   "DT_RMSE":   r["DT"]["RMSE"],   "DT_MAPE":   r["DT"]["MAPE"],
            "Naive_MAE": r["Naive"]["MAE"], "Naive_RMSE": r["Naive"]["RMSE"], "Naive_MAPE": r["Naive"]["MAPE"],
        })
    return pd.DataFrame(rows)

def wilcoxon_lstm_vs_gbm(df):
    lstm_rmse = df["LSTM_RMSE"].values
    gbm_rmse  = df["GBM_RMSE"].values
    stat, p_value = wilcoxon(lstm_rmse, gbm_rmse, alternative="less")
    n_lstm_wins = int(np.sum(lstm_rmse < gbm_rmse))
    print("\n" + "═" * 60)
    print("WILCOXON SIGNED-RANK TEST  —  LSTM vs GBM (RMSE)")
    print("═" * 60)
    print(f"  Tickers where LSTM < GBM : {n_lstm_wins} / {len(lstm_rmse)}")
    print(f"  Test statistic (W)       : {stat:.4f}")
    print(f"  p-value (one-sided)      : {p_value:.6f}")
    sig = "YES — statistically significant" if p_value < 0.05 else "NO"
    print(f"  Significant at alpha=0.05 : {sig}")
    print("═" * 60)
    return {"statistic": float(stat), "p_value": float(p_value), "lstm_wins": n_lstm_wins}


results = run_all()
df = results_to_dataframe(results)
print("\nFULL RESULTS TABLE (averaged across walk-forward windows):\n")
print(df.to_string(index=False))
df.to_csv("results_walkforward.csv", index=False)
print("\nSaved → results_walkforward.csv")
wilcoxon_lstm_vs_gbm(df)

Evaluating NKE ...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  NKE  window 1/3 done


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  NKE  window 2/3 done


...

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  AMD  window 2/3 done


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  AMD  window 3/3 done

FULL RESULTS TABLE (averaged across walk-forward windows):

Ticker  LSTM_MAE  LSTM_RMSE  LSTM_MAPE  GBM_MAE  GBM_RMSE  GBM_MAPE   DT_MAE  DT_RMSE  DT_MAPE  Naive_MAE  Naive_RMSE  Naive_MAPE
   NKE    2.1944     2.6999     2.6624   7.6642    7.9449    8.6372   2.9785   3.7759   3.7143     1.0586      1.6748      1.3159
  AAPL   13.6516    15.5685     5.5667  48.5816   49.0376   21.3745  53.8903  54.9120  22.0915     2.3989      3.3284      1.0175
  NVDA   40.1526    40.3256    28.7645  56.5950   57.0539   47.6327  88.7098  88.8324  63.3079     2.3211      2.9103      1.8218
   JNJ    7.6794     8.4589     3.7323  10.6637   11.0153    6.6209  17.8711  18.8832   8.7668     1.3988      1.7810      0.8257
   XOM    6.6005     7.0479     5.6225   2.4113    2.8994    2.1889  26.7670  27.7620  25.3059     1.2841      1.5823      1.1060
   JPM   20.7345    21.6852     7.2750  20.4984   21.1767    9.0611  85.0647  86.0137  31.0929     2.4851      3.1869      0.9757
  MSFT

{'statistic': 33.0, 'p_value': 0.0014286041259765625, 'lstm_wins': 17}

In [ ]:

def get_gbm_predictions(params, test_prices, context_last_price):
    preds = []
    for i in range(len(test_prices)):
        price = context_last_price if i == 0 else test_prices[i - 1]
        p = {**params, "lastPrice": price}
        median = forecast_gbm(p, horizon=1, n_paths=500, seed=42 + i)["median"]
        preds.append(median[0])
    return np.array(preds)

def get_lstm_predictions(model, scaler, test_prices, context_prices, lookback=LOOKBACK):
    full = list(context_prices[-lookback:]) + list(test_prices)
    full_scaled = scaler.transform(full)
    x_test, _ = build_sequences(full_scaled, lookback)
    preds_scaled = model.predict(x_test, verbose=0).flatten()
    return np.array(scaler.inverse(preds_scaled.tolist()))

def get_dt_predictions(tree, test_prices, test_volumes):
    feats = compute_features(test_prices, test_volumes)
    preds, actuals = [], []
    for i in range(len(feats) - 1):
        preds.append(tree.predict([feature_vector(feats[i])])[0])
        actuals.append(test_prices[i + 1])
    return np.array(preds), np.array(actuals)


def evaluate_ensemble(gbm_preds, lstm_preds, dt_preds, actuals):
    # Align lengths — DT loses one sample due to feature construction
    n = min(len(gbm_preds), len(lstm_preds), len(dt_preds), len(actuals))
    ensemble = (gbm_preds[-n:] + lstm_preds[-n:] + dt_preds[-n:]) / 3.0
    return _metrics(ensemble, actuals[-n:])

def walk_forward_with_ensemble(ticker):
    prices, volumes = fetch_stock_data(ticker)
    n = len(prices)

    last_region_start = int(n * 0.80)
    latest_start = n - HORIZON
    window_starts = np.linspace(last_region_start, latest_start,
                                N_WINDOWS, dtype=int).tolist()

    per_window = {"GBM": [], "LSTM": [], "DT": [], "Naive": [], "Ensemble": []}

    for w_idx, test_start in enumerate(window_starts):
        test_end = test_start + HORIZON
        hist_prices  = prices[:test_start]
        hist_volumes = volumes[:test_start]
        test_prices  = prices[test_start:test_end]

        tr_p, val_p, _ = time_split(hist_prices)
        tr_v, val_v, _ = time_split(hist_volumes)
        context_last_price = hist_prices[-1]

        gbm_params     = fit_gbm(tr_p)
        model, scaler  = train_lstm(tr_p, val_p)
        tree           = train_decision_tree(tr_p, tr_v, val_p, val_v)

        gbm_preds  = get_gbm_predictions(gbm_params, test_prices, context_last_price)

        lstm_preds = get_lstm_predictions(model, scaler, test_prices, hist_prices)

        dt_test_prices  = np.concatenate([hist_prices[-40:], test_prices])
        dt_test_volumes = np.concatenate([hist_volumes[-40:],
                                          volumes[test_start:test_end]])
        dt_preds, dt_actuals = get_dt_predictions(tree, dt_test_prices, dt_test_volumes)

        per_window["GBM"].append(_metrics(gbm_preds, test_prices))
        per_window["LSTM"].append(_metrics(
            lstm_preds[-len(dt_actuals):], test_prices[-len(dt_actuals):]))
        per_window["DT"].append(_metrics(dt_preds, dt_actuals))
        per_window["Naive"].append(evaluate_naive(test_prices, context_last_price))

        per_window["Ensemble"].append(
            evaluate_ensemble(gbm_preds, lstm_preds, dt_preds, dt_actuals))

        print(f"  {ticker}  window {w_idx+1}/{N_WINDOWS} done")

    def _avg(list_of_dicts):
        return {m: round(float(np.mean([d[m] for d in list_of_dicts])), 4)
                for m in ["MAE", "RMSE", "MAPE"]}

    return {mdl: _avg(per_window[mdl]) for mdl in per_window}



def run_all_with_ensemble():
    results = {}
    for ticker in TICKERS:
        print(f"Evaluating {ticker} ...")
        try:
            results[ticker] = walk_forward_with_ensemble(ticker)
        except Exception as e:
            print(f"  !! {ticker} failed: {e}")
    return results

def results_with_ensemble_to_df(results):
    rows = []
    for ticker, r in results.items():
        rows.append({
            "Ticker":         ticker,
            "LSTM_RMSE":      r["LSTM"]["RMSE"],   "LSTM_MAPE":  r["LSTM"]["MAPE"],
            "GBM_RMSE":       r["GBM"]["RMSE"],    "GBM_MAPE":   r["GBM"]["MAPE"],
            "DT_RMSE":        r["DT"]["RMSE"],      "DT_MAPE":    r["DT"]["MAPE"],
            "Ensemble_RMSE":  r["Ensemble"]["RMSE"],"Ensemble_MAPE": r["Ensemble"]["MAPE"],
            "Naive_RMSE":     r["Naive"]["RMSE"],   "Naive_MAPE": r["Naive"]["MAPE"],
        })
    return pd.DataFrame(rows)


results_ens = run_all_with_ensemble()
df_ens = results_with_ensemble_to_df(results_ens)

print("\nFULL RESULTS WITH ENSEMBLE (RMSE + MAPE, walk-forward averaged):\n")
print(df_ens.to_string(index=False))
df_ens.to_csv("results_with_ensemble.csv", index=False)
print("\nSaved → results_with_ensemble.csv")

print("\nPer-ticker winner by RMSE:")
cols = ["LSTM_RMSE","GBM_RMSE","DT_RMSE","Ensemble_RMSE","Naive_RMSE"]
labels = ["LSTM","GBM","DT","Ensemble","Naive"]
for _, row in df_ens.iterrows():
    vals = [row[c] for c in cols]
    winner = labels[np.argmin(vals)]
    print(f"  {row['Ticker']:6s}  winner: {winner}  "
          f"(LSTM={row['LSTM_RMSE']:.2f}, Ensemble={row['Ensemble_RMSE']:.2f}, "
          f"Naive={row['Naive_RMSE']:.2f})")

Evaluating NKE ...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  NKE  window 1/3 done


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  NKE  window 2/3 done


...

  AMD  window 1/3 done


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  AMD  window 2/3 done


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


  AMD  window 3/3 done

FULL RESULTS WITH ENSEMBLE (RMSE + MAPE, walk-forward averaged):

Ticker  LSTM_RMSE  LSTM_MAPE  GBM_RMSE  GBM_MAPE  DT_RMSE  DT_MAPE  Ensemble_RMSE  Ensemble_MAPE  Naive_RMSE  Naive_MAPE
   NKE     2.4582     2.3426    7.8744    8.6372   3.5383   3.4707         4.0164         4.1834      1.6599      1.3159
  AAPL    13.3732     4.6909   49.0376   21.3745  56.2762  22.6459        12.3313         4.8528      3.3284      1.0175
  NVDA    23.0320    16.2566   56.9875   47.6326  88.7902  63.3778        19.7005        13.0400      2.9069      1.8218
   JNJ     7.0095     3.3375   11.0152    6.6209  19.2536   8.9271         9.7277         4.6958      1.7810      0.8257
   XOM     5.4065     4.0102    2.8994    2.1889  26.4701  23.9156        10.2055         8.9335      1.5823      1.1060
   JPM    24.3824     9.0396   21.1767    9.0611  86.1527  31.1444        32.2220        11.5918      3.1869      0.9757
  MSFT    21.1583     4.3458   95.1729   23.0557  98.9661  22.0